
# TableCoT experiments

In [ ]:
!git clone https://github.com/KevinHuang8/Augment_QA.git
!pip install -q openai tenacity pyrootutils datasets==2.14.7 \
    transformers pandas recognizers-text-suite func_timeout \
    fuzzywuzzy records emoji==1.7.0
!pip install -q flash-attn --no-build-isolation
!pip install -q vllm==0.6.6.post1

In [ ]:
%cd /content/Augment_QA

In [ ]:
import subprocess, time, requests
import os
# os.environ["VLLM_ATTENTION_BACKEND"] = "XFORMERS"

process = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "Qwen/Qwen2.5-3B-Instruct",
     "--port", "8000",
     "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.9",
     "--dtype", "half",
    #  "--enforce-eager"],           # disables CUDA graphs + uses eager mode, avoids FA2
    #  "--attention-backend", "xformers"
     ],
    stdout=open("vllm.log", "w"),
    stderr=subprocess.STDOUT
)

for i in range(36):
    try:
        r = requests.get("http://localhost:8000/v1/models")
        if r.status_code == 200:
            print(f"Server ready after {(i+1)*5}s!")
            break
    except:
        pass
    time.sleep(5)
else:
    print("Server didn't start. Checking logs:")
    !cat vllm.log | tail -30

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")
resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)
print(resp.choices[0].message.content)


In [ ]:
# Table-CoT: FinQA
import time
start = time.time()
print("Beginning of run:")
print(start)
%cd /content/Augment_QA/TableCoT/finqa
!git pull origin main
!python prompt.py --start 0 --end 158 --option cot --model Qwen/Qwen2.5-3B-Instruct
!python compute_score.py --inputs "outputs/response_*cot*.json"
print(f"Elapsed: {(time.time() - start)/60:.1f} minutes")

In [ ]:
# Table-CoT: WikiTQ
import time
start = time.time()
print("Beginning of run:")
%cd /content/Augment_QA/TableCoT/wikitableqa
!python prompt.py --start 0 --end 1000 --option cot --model Qwen/Qwen2.5-3B-Instruct

%cd /content/Augment_QA/TableCoT/wikitableqa/outputs
!python postprocess_answer.py --inputs "response_*Qwen*.json"

!python compute_score.py --inputs "response_*Qwen*.json.processed"
print(f"Elapsed: {(time.time() - start)/60:.1f} minutes")


In [ ]:
# Table-CoT: TATQA
import time
start = time.time()
print("Beginning of run:")
%cd /content/Augment_QA/TableCoT/tatqa
!python prompt.py --start 0 --end 1668 --option cot --model Qwen/Qwen2.5-3B-Instruct
!python compute_score.py --inputs "outputs/response_*cot*.json"
print(f"Elapsed: {(time.time() - start)/60:.1f} minutes")


In [ ]:
# Table-CoT: WikiTQ 8-shot (beyond paper experiment)
import time
start = time.time()
print("Beginning of run:")
%cd ..
%cd /content/Augment_QA/TableCoT/wikitableqa
!python prompt_extend.py --start 0 --end 1000 --option cot --model Qwen/Qwen2.5-3B-Instruct --shots 4
!python prompt_extend.py --start 0 --end 1000 --option cot --model Qwen/Qwen2.5-3B-Instruct --shots 8

%cd /content/Augment_QA/TableCoT/wikitableqa/outputs
!python postprocess_answer.py --inputs "response_*Qwen*.json"

!python compute_score.py --inputs "response_*Qwen*.json.processed"
print(f"Elapsed: {(time.time() - start)/60:.1f} minutes")


In [ ]:
from google.colab import files
import glob

result_files = (
    glob.glob('/content/Augment_QA/TableCoT/*/outputs/response_*.json') +
    glob.glob('/content/Augment_QA/TableCoT/wikitableqa/outputs/*.processed')
)
for f in result_files:
    files.download(f)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>